# Tariikhna — Automatic Passage Extractor

Extracts complete passages from the Sira book by detecting bold headings.
Each passage = one full section (heading + all body text until the next heading).

This produces realistic, full-context inputs for the data pipeline - exactly the kind
of input the model will see in production.

Output format matches the pipeline's expected input:
```json
[{"passage_id": "...", "heading": "...", "passage": "full text", "source": "...", "page": N}]
```

## 1. Setup

In [ ]:
!pip install pdfplumber -q

In [ ]:
import pdfplumber
import json
import re
import random

# ============================================
# SET YOUR PDF PATH
# ============================================
PDF_PATH = 'Mohamed_Man_and_Prophet_Adil_Salahi.pdf'

# Content boundaries (skip front matter, bibliography, index)
CONTENT_START_PAGE = 25   # narrative begins around here
CONTENT_END_PAGE = 389    # before Bibliography

BOOK_SOURCE = 'Adil Salahi, Muhammad: Man and Prophet'

print('Setup ready!')

## 2. The Extractor

Detects bold headings, captures the full body text under each one.

In [ ]:
def clean_text(text):
    """Clean up transliteration spacing artifacts and footnote markers."""
    text = re.sub(r'(?<=[a-z])\.\d+(?=\s|$)', '.', text)  # remove footnote markers like .12
    text = re.sub(r'\s+', ' ', text)  # normalize whitespace
    # Fix transliteration breaks: letter space diacritic
    text = re.sub(r'([A-Za-z])\s+([ūīāṣḥṭẓḍ\u0100-\u017f\u1e00-\u1eff])', r'\1\2', text)
    text = re.sub(r'([ūīāṣḥṭẓḍ\u0100-\u017f\u1e00-\u1eff])\s+([a-z])', r'\1\2', text)
    # Fix missing space before capitalized diacritic names (thatAminah -> that Aminah)
    text = re.sub(r'([a-z])([ĀĪŪṢḤṬẒḌ])', r'\1 \2', text)
    return text.strip()

def normalize_heading(text):
    """Collapse spaced-out caps like 'N OTES' to detect them."""
    return re.sub(r'\s+', '', text).upper()

def is_heading_line(line_words, line_text):
    """Decide if a line is a bold section heading."""
    if not line_words:
        return False
    bold_count = sum(1 for w in line_words if 'Bold' in w.get('fontname', ''))
    txt = line_text.strip()
    norm = normalize_heading(txt)
    # Heading criteria: mostly bold, short, not 'NOTES' (catches 'N OTES'), not a footnote
    return (bold_count >= len(line_words) * 0.7 and
            len(line_words) <= 10 and
            len(txt) > 4 and
            'NOTES' not in norm and
            not txt[0].isdigit())

def extract_all_sections(pdf_path, start_page, end_page):
    """Extract all heading->body sections from the PDF."""
    pdf = pdfplumber.open(pdf_path)
    sections = []
    current_heading = None
    current_body = []
    current_start_page = start_page
    skip_until_next_heading = False
    
    for page_num in range(start_page, min(end_page, len(pdf.pages))):
        page = pdf.pages[page_num]
        words = page.extract_words(extra_attrs=['fontname', 'size'], keep_blank_chars=False)
        
        # Group words into lines by vertical position
        lines = {}
        for w in words:
            top = round(w['top'])
            bucket = None
            for existing in lines:
                if abs(existing - top) < 5:
                    bucket = existing
                    break
            if bucket is None:
                bucket = top
                lines[bucket] = []
            lines[bucket].append(w)
        
        for top in sorted(lines.keys()):
            line_words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = ' '.join(w['text'] for w in line_words)
            bold_count = sum(1 for w in line_words if 'Bold' in w.get('fontname', ''))
            norm = normalize_heading(line_text.strip())
            
            # Detect a NOTES heading -> skip its body (footnotes/citations)
            is_notes = (bold_count >= len(line_words)*0.7 and 'NOTES' in norm and len(line_words) <= 3)
            if is_notes:
                if current_heading and current_body:
                    body = clean_text(' '.join(current_body))
                    sections.append({'heading': current_heading, 'body': body, 'page': current_start_page})
                current_heading = None
                current_body = []
                skip_until_next_heading = True
                continue
            
            if is_heading_line(line_words, line_text):
                if current_heading and current_body:
                    body = clean_text(' '.join(current_body))
                    sections.append({'heading': current_heading, 'body': body, 'page': current_start_page})
                current_heading = clean_text(line_text)
                current_body = []
                current_start_page = page_num + 1
                skip_until_next_heading = False
            else:
                if current_heading and not skip_until_next_heading:
                    current_body.extend(w['text'] for w in line_words)
    
    if current_heading and current_body:
        body = clean_text(' '.join(current_body))
        sections.append({'heading': current_heading, 'body': body, 'page': current_start_page})
    
    pdf.close()
    return sections

print('Extractor ready!')

## 3. Extract All Sections

In [ ]:
all_sections = extract_all_sections(PDF_PATH, CONTENT_START_PAGE, CONTENT_END_PAGE)
print(f'Extracted {len(all_sections)} total sections\n')

# Show word count distribution
word_counts = [len(s['body'].split()) for s in all_sections]
print(f'Word count stats:')
print(f'  Min: {min(word_counts)}, Max: {max(word_counts)}')
print(f'  Average: {sum(word_counts)/len(word_counts):.0f}')

print(f'\nFirst 15 section headings:')
for s in all_sections[:15]:
    wc = len(s['body'].split())
    print(f'  [{wc:4d}w] {s["heading"]}')

## 4. Filter to Good Passages

Keep passages in a reasonable length range with narrative content.

In [ ]:
# ============================================
# FILTER SETTINGS
# ============================================
MIN_WORDS = 150   # skip tiny fragments
MAX_WORDS = 1500  # skip overly long sections (can split later if needed)

def is_narrative(section):
    """Heuristic: does this section contain narrative action?"""
    body = section['body'].lower()
    # Should mention people doing things
    action_words = ['said', 'went', 'came', 'asked', 'told', 'walked', 'prophet',
                    'replied', 'gave', 'took', 'saw', 'met', 'prayed', 'fought']
    return sum(1 for w in action_words if w in body) >= 3

good_passages = [
    s for s in all_sections
    if MIN_WORDS <= len(s['body'].split()) <= MAX_WORDS
    and is_narrative(s)
]

print(f'{len(good_passages)} passages pass filters (out of {len(all_sections)} total)')
print(f'\nLength distribution of good passages:')
wc = [len(s["body"].split()) for s in good_passages]
import statistics
print(f'  Min: {min(wc)}, Max: {max(wc)}, Median: {statistics.median(wc):.0f}')

## 5. Random Sampling for Consistency

Pick a random sample (with a fixed seed for reproducibility).

In [ ]:
# ============================================
# SAMPLING SETTINGS
# ============================================
SAMPLE_SIZE = 40       # how many passages to extract for training
RANDOM_SEED = 42       # fixed seed = same sample every run (reproducible)

random.seed(RANDOM_SEED)

if SAMPLE_SIZE >= len(good_passages):
    sampled = good_passages
    print(f'Taking all {len(sampled)} passages (sample size >= available)')
else:
    sampled = random.sample(good_passages, SAMPLE_SIZE)
    print(f'Randomly sampled {SAMPLE_SIZE} passages (seed={RANDOM_SEED})')

# Sort by page order for readability
sampled.sort(key=lambda s: s['page'])

print(f'\nSampled passages:')
for s in sampled:
    wc = len(s['body'].split())
    print(f'  p{s["page"]:3d} [{wc:4d}w] {s["heading"]}')

## 6. Format for the Pipeline

In [ ]:
def make_passage_id(heading, page):
    """Create a clean passage_id from the heading."""
    slug = re.sub(r'[^a-z0-9]+', '_', heading.lower()).strip('_')
    slug = slug[:40]  # limit length
    return f'{slug}_p{page}'

pipeline_input = []
for s in sampled:
    pipeline_input.append({
        'passage_id': make_passage_id(s['heading'], s['page']),
        'heading': s['heading'],
        'passage': s['body'],
        'source': f'{BOOK_SOURCE}, p. {s["page"]}',
        'page': s['page']
    })

# Save
with open('extracted_passages.json', 'w', encoding='utf-8') as f:
    json.dump(pipeline_input, f, ensure_ascii=False, indent=2)

print(f'Saved {len(pipeline_input)} passages to extracted_passages.json')
print('\nThis file is ready to feed into the data pipeline!')

## 7. Preview a Few Full Passages

In [ ]:
# Show 3 complete passages to verify quality
for p in pipeline_input[:3]:
    print('='*70)
    print(f'HEADING: {p["heading"]}')
    print(f'ID: {p["passage_id"]}')
    print(f'SOURCE: {p["source"]}')
    print(f'LENGTH: {len(p["passage"].split())} words')
    print('-'*70)
    print(p['passage'])
    print()

## 8. (Optional) Browse All Headings

See every section in the book to manually pick specific ones if you prefer.

In [ ]:
print('ALL narrative sections in the book:')
print('='*70)
for s in good_passages:
    wc = len(s['body'].split())
    print(f'  p{s["page"]:3d} [{wc:4d}w] {s["heading"]}')

# To manually pick specific headings instead of random:
# wanted = ['The Hijra', 'A New Tragedy', 'Bakr\'s Noble Heart']
# manual = [s for s in good_passages if any(w in s['heading'] for w in wanted)]
# Then format manual the same way as above

## 9. Download

In Colab:
```python
from google.colab import files
files.download('extracted_passages.json')
```

Then feed `extracted_passages.json` into the production data pipeline as the INPUT_FILE.